%md

###### 19_nlp_mlflow_tracking

###### Purpose
Validate the MLflow experiment, inspect logged runs and artifacts, verify the best model, and understand the contents of the saved MLflow model before registration.

###### Technologies Used

- Databricks
- Unity Catalog
- MLflow
- Scikit-learn
- TF-IDF
- Delta Lake

###### Input
- MLflow experiment path
- Logged MLflow runs
- best_model_info Delta table
- Logged MLflow model (models:/...)

######  Output
- Verified MLflow experiment
- Displayed experiment metadata
- Compared all model runs
- Validated best model information
- Inspected logged model artifacts
- Verified model signature and pipeline contents

######  Architecture

```text

Current User
      ↓
Build Experiment Path
      ↓
Load MLflow Experiment
      ↓
Display Experiment Metadata
      ↓
Search Runs
      ↓
Compare Metrics
      ↓
Validate best_model_info
      ↓
Load Logged Model
      ↓
Inspect Pipeline
      ↓
Inspect Signature

```

###### Section 1: Validate MLflow Experiment and Artifacts

In [0]:
import mlflow

#Get current user
user_name = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .userName()
    .get()
)

#Build experiment path
experiment_path = f"/Users/{user_name}/support_ticket_experiment"
experiment = mlflow.get_experiment_by_name(experiment_path)

#safety check in case the experiment does not exist
if experiment is None:
    raise ValueError(f"Experiment not found: {experiment_path}")

#Print experiment metadata
print("Experiment name:", experiment.name)
print("Experiment ID:", experiment.experiment_id)
print("Artifact location:", experiment.artifact_location)

#Search runs ordered by f1_macro
runs_df = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.f1_macro DESC"]
)
display(runs_df)

#Display best_model_info
best_model_info = spark.table("dbw_agentic_ai_dev.support_ticket_ai.best_model_info")
display(best_model_info)

###### Section 2 : Inspect Logged MLflow Model Artifacts

In [0]:
# to inspect the model.pkl, MLmodel, signature, TF-IDF vocabulary, classifier

import mlflow
from mlflow.models import Model

# Model URI copied from the registered MLflow model
model_uri = 'models:/m-694867e1c0f140a2adb89d9b7ace62ce'

loaded_model = mlflow.sklearn.load_model(model_uri)

#To inspect the loaded pipeline
print(type(loaded_model))
print(loaded_model)
print(loaded_model.named_steps)
print(loaded_model.named_steps["tfidf"].vocabulary_)
print(loaded_model.named_steps["classifier"])

tfidf = loaded_model.named_steps["tfidf"]
classifier = loaded_model.named_steps["classifier"]

print(type(tfidf))
print(type(classifier))
print(len(tfidf.vocabulary_))
print(list(tfidf.vocabulary_.keys())[:20])

#To inspect signature
model_info = Model.load(model_uri)
print(model_info.signature)


###### Notebook Summary

Notebook Summary

- Validated the MLflow experiment path.
- Displayed experiment metadata.
- Compared all logged model runs.
- Verified the selected best model.
- Inspected the serialized sklearn pipeline.
- Explored TF-IDF vocabulary and classifier.
- Verified the model signature.


######  Key Learnings
- model.pkl stores the trained Scikit-learn pipeline (TF-IDF + classifier).
- MLmodel is the metadata file that tells MLflow how to load the model.
- Model Signature defines expected inputs and outputs for deployment.
- Input Example helps validate the model before serving.
- MLflow artifacts contain everything required to reproduce the trained model.
- A registered model is created from MLflow artifacts, not by retraining.





%md

###### Next Notebook

20_nlp_model_registry

Register the best Support Ticket NLP classification model from MLflow into Unity Catalog Model Registry.